# Load data: Phai load duoc Common

In [1]:
import sys
sys.path.append('../Common')

import CommonYFinance

In [ ]:
def detectSignal(symbol, from_date, to_date, timeframe):

    import pandas as pd
    import plotly.graph_objects as go
    import redis
    import numpy as np
    import talib 

    from datetime import datetime
    from statsmodels.tsa.arima.model import ARIMA

    # ##############################################Step 1: Lấy dữ liệu##############################################
    data = CommonYFinance.CommonYFinance.loaddataYFinance(symbol, from_date, to_date, timeframe)

    # ##############################################Step 2: Chiến lược##############################################  
    # Giả sử `data` là DataFrame chứa các cột Datetime, Open, High, Low, Close, Volume
    # Đặt chỉ mục DataFrame với cột 'Datetime' có dữ liệu kiểu datetime
    data.set_index('Datetime', inplace=True)
    # data.index = pd.DatetimeIndex(data.index, freq='min')  # Mỗi phút

    # Tính Momentum
    data['Momentum'] = talib.MOM(data['Close'], timeperiod=1)
    # Tính RSI    
    data['RSI'] = talib.RSI(data['Close'], timeperiod=14)

    # Chọn tham số ARIMA dựa trên AIC, BIC hoặc các tiêu chí khác (cần phải thực hiện trước) => Chay cell o tren
    model = ARIMA(data['Close'], order=(2, 5, 3))
    model_fit = model.fit()

    # Dự đoán giá
    data['Predicted_Close'] = model_fit.predict(start=0, end=len(data)-1, typ='levels')

    # Xác định tín hiệu mua và bán
    data['Buy_Signal'] = ((data['Predicted_Close'] > data['Close']) & (data['Momentum'] > 0) & (data['RSI'] < 30))
    data['Sell_Signal'] = ((data['Predicted_Close'] < data['Close']) & (data['Momentum'] < 0) & (data['RSI'] > 70))
    # data['Sell_Signal'] = True

    # ##############################################Step 3: Đẩy dữ liệu qua Redis##############################################
    # Nếu có tín hiệu thì đẩy qua Redis
    # Tạo kết nối Redis
    r = redis.Redis(host='localhost', port=6379, db=9)

    # Tạo hash key
    hash_key = 'OG_CK_Arima, Momentum, RSI_K11' + symbol
    last_record = data.iloc[-1] # Lay du lieu moi nhat
    print(last_record)
    
    # Chuyển đổi record cuối cùng thành từ điển và lưu vào Redis dưới dạng hash
    if(last_record['Buy_Signal'] == True or last_record['Sell_Signal'] == True):
        for field, value in last_record.to_dict().items():
            # Chuyển đổi giá trị uint64 và Timestamp thành chuỗi
            if isinstance(value, pd.Timestamp):
                value = value.isoformat()
            elif isinstance(value, (int, np.uint64)):
                value = str(value)
            r.hset(hash_key, field, value)  
            r.hset(hash_key, 'Symbol', symbol)
            r.hset(hash_key, 'Insertdate', datetime.now().isoformat())
        print(last_record)   
    else:
        print(last_record)
        print('Không có tín hiệu!')   

# Schedule

In [3]:
# from datetime import datetime, timedelta
# import schedule
# import time

# def schedule_detectSignal():
#     # Cac khai bao Symbol, Date, Timeframe
#     symbol = 'VCB.VN'
#     from_date = (datetime.now() - timedelta(days=0)).strftime('%Y-%m-%d')  # Lấy ngày hôm qua
#     to_date = (datetime.now() + timedelta(days=1)).strftime('%Y-%m-%d')
#     timeframe = '1d' # 1m
#     detectSignal(symbol, from_date, to_date, timeframe)

# # Task chay: Viet theo While True
# # Danh sách các phút cụ thể bạn muốn hàm được chạy
# # run_minutes = [17, 27, 50, 2]
# run_minutes = list(range(0, 60, 1)) # Chien luoc 5m chay 1 lan
# last_run = None

# while True:
#     current_time = datetime.now()
#     current_minute = current_time.minute
#     # Kiểm tra xem phút hiện tại có nằm trong danh sách các phút cần chạy hàm không
#     if current_minute in run_minutes:
#         time.sleep(1)
#         # Kiểm tra xem đã chạy hàm trong phút hiện tại hay chưa
#         last_run = getattr(schedule_detectSignal, 'last_run', None)
#         if last_run is None or last_run != current_minute:
#             schedule_detectSignal()
#             # Lưu lại phút cuối cùng mà hàm đã chạy
#             setattr(schedule_detectSignal, 'last_run', current_minute)   

#     # Nghỉ 1 giây trước khi kiểm tra lại
#     time.sleep(1)

# Chien luoc that 1 ngay 1 lan, vao luc 06:00

In [ ]:
from datetime import datetime, timedelta
import schedule
import time

def schedule_detectSignal():
    # Cac khai bao Symbol, Date, Timeframe
    symbol = 'VCB.VN'
    from_date = (datetime.now() - timedelta(days=30)).strftime('%Y-%m-%d')  # Lấy ngày hôm qua
    to_date = (datetime.now() + timedelta(days=1)).strftime('%Y-%m-%d')
    timeframe = '1d' # 1m
    detectSignal(symbol, from_date, to_date, timeframe)

# Lên lịch hàm scan_market để chạy mỗi ngày
schedule.every(1).day.at("21:54:00").do(schedule_detectSignal)

while True:
	schedule.run_pending() # Hàm này được gọi liên tục để kiểm tra xem có công việc nào đã đến thời gian cần thực hiện hay không
	# time.sleep(1)